# 구글 코랩 사용

## 0단계. 기본 설치 및 라이브러리
프로젝트 실행에 필요한 패키지를 설치하고, 데이터 수집·전처리·분석에 필요한 라이브러리를 불러온다.  
특히 Kiwi 형태소 분석기를 사용하기 위해 `kiwipiepy`를 설치한다.

In [18]:
# =========================================================
# 0. 기본 설치 및 라이브러리
# 버전 2: Kiwi 형태소 분석기 사용
# =========================================================

!pip install -q requests beautifulsoup4 lxml gensim statsmodels scipy scikit-learn kiwipiepy

import os
import re
import time
import zipfile
import requests
import numpy as np
import pandas as pd

from io import BytesIO
from bs4 import BeautifulSoup

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from gensim.models import FastText

from scipy.stats import spearmanr, ttest_ind, mannwhitneyu

import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel

from kiwipiepy import Kiwi

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

## 1단계. 구글 드라이브 연결 및 저장 경로 설정
Colab 런타임이 끊겨도 중간 결과를 유지할 수 있도록 Google Drive를 연결한다.  
이후 생성되는 체크포인트 파일과 최종 전처리 결과는 `SAVE_DIR`에 저장된다.

In [19]:
# =========================================================
# 1. 구글 드라이브 연결 및 저장 경로 설정
# =========================================================

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/esg_dart_project_v2"
os.makedirs(SAVE_DIR, exist_ok=True)

print("저장 경로:", SAVE_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
저장 경로: /content/drive/MyDrive/esg_dart_project_v2


## 2단계. OpenDART API KEY 입력
DART API를 호출하기 위해 OpenDART 인증키를 입력받는다.  
API key를 코드에 직접 적지 않고 `input()`으로 입력받아 보안 위험을 줄였다.

In [ ]:
# =========================================================
# 2. OpenDART API KEY 입력
# =========================================================

OPENDART_API_KEY = input("OpenDART API KEY를 입력하세요: ").strip()

## 3단계. company_master.csv 업로드 및 기본 확인
분석 대상 기업-연도 데이터인 `company_master.csv`를 업로드한다.  
`stock_code`를 6자리 문자열로 맞추고, `esg_year`를 생성하며, ESG 등급을 숫자형 변수로 변환한다.

In [21]:
# =========================================================
# 3. company_master.csv 업로드 및 기본 확인
# =========================================================

from google.colab import files

uploaded = files.upload()
csv_name = list(uploaded.keys())[0]

master = pd.read_csv(csv_name, dtype={"stock_code": str})
master["stock_code"] = master["stock_code"].astype(str).str.zfill(6)

if "esg_year" not in master.columns and "fiscal_year" in master.columns:
    master["esg_year"] = master["fiscal_year"] + 1

grade_map = {
    "S": 6,
    "A+": 5,
    "A": 4,
    "B+": 3,
    "B": 2,
    "C": 1,
    "D": 0
}

for col in ["esg_grade", "e_grade", "s_grade", "g_grade"]:
    if col in master.columns:
        master[col + "_num"] = master[col].map(grade_map)

print(master.shape)
display(master.head())
display(master.columns)

Saving company_master.csv to company_master.csv
(381, 17)


,company_name,corp_code,stock_code,industry,fiscal_year,report_code,rcept_no,esg_source,esg_grade,e_grade,s_grade,g_grade,esg_year,esg_grade_num,e_grade_num,s_grade_num,g_grade_num
0,삼성전자,NaN,005930,전기전자,2022,11011,NaN,한국ESG기준원,A,A,A+,B+,2023,4,4,5,3
1,삼성전자,NaN,005930,전기전자,2023,11011,NaN,한국ESG기준원,B+,B+,A,B,2024,3,3,4,2
2,삼성전자,NaN,005930,전기전자,2024,11011,NaN,한국ESG기준원,A,B+,A+,B+,2025,4,3,5,3
3,BYC,NaN,001460,섬유/의류,2022,11011,NaN,한국ESG기준원,D,D,D,D,2023,0,0,0,0
4,BYC,NaN,001460,섬유/의류,2023,11011,NaN,한국ESG기준원,D,D,D,C,2024,0,0,0,1


Index(['company_name', 'corp_code', 'stock_code', 'industry', 'fiscal_year',
       'report_code', 'rcept_no', 'esg_source', 'esg_grade', 'e_grade',
       's_grade', 'g_grade', 'esg_year', 'esg_grade_num', 'e_grade_num',
       's_grade_num', 'g_grade_num'],
      dtype='object')

## 4단계. corp_code 매핑 파일 다운로드
OpenDART의 `corpCode.xml`을 이용해 `stock_code`와 `corp_code`를 연결한다.  
이미 저장된 `corp_code_mapping.csv`가 있으면 다시 다운로드하지 않고 재사용한다.

In [22]:
# =========================================================
# 4. corp_code 매핑 파일 다운로드
# 체크포인트 저장 + 재사용 버전
# =========================================================

CORP_CODE_PATH = os.path.join(SAVE_DIR, "corp_code_mapping.csv")

def download_corp_code(api_key):
    url = "https://opendart.fss.or.kr/api/corpCode.xml"

    res = requests.get(
        url,
        params={"crtfc_key": api_key},
        timeout=30
    )
    res.raise_for_status()

    with zipfile.ZipFile(BytesIO(res.content)) as zf:
        xml_name = zf.namelist()[0]
        xml_text = zf.read(xml_name).decode("utf-8", errors="ignore")

    soup = BeautifulSoup(xml_text, "xml")

    rows = []

    for item in soup.find_all("list"):
        rows.append({
            "corp_code": item.find("corp_code").get_text(strip=True) if item.find("corp_code") else None,
            "corp_name_dart": item.find("corp_name").get_text(strip=True) if item.find("corp_name") else None,
            "stock_code": item.find("stock_code").get_text(strip=True) if item.find("stock_code") else None,
            "modify_date": item.find("modify_date").get_text(strip=True) if item.find("modify_date") else None
        })

    corp_df = pd.DataFrame(rows)

    corp_df["stock_code"] = corp_df["stock_code"].astype(str).str.zfill(6)
    corp_df["corp_code"] = corp_df["corp_code"].astype(str).str.zfill(8)

    corp_df = corp_df[corp_df["stock_code"].notna()].copy()
    corp_df = corp_df[corp_df["stock_code"] != "000000"].copy()

    return corp_df


if os.path.exists(CORP_CODE_PATH):
    print("기존 corp_code_mapping.csv 불러오기")

    corp_df = pd.read_csv(
        CORP_CODE_PATH,
        dtype={"stock_code": str, "corp_code": str}
    )

    corp_df["stock_code"] = corp_df["stock_code"].astype(str).str.zfill(6)
    corp_df["corp_code"] = corp_df["corp_code"].astype(str).str.zfill(8)

else:
    print("OpenDART에서 corp_code 새로 다운로드")

    corp_df = download_corp_code(OPENDART_API_KEY)

    corp_df.to_csv(
        CORP_CODE_PATH,
        index=False,
        encoding="utf-8-sig"
    )

print(corp_df.shape)
display(corp_df.head())

기존 corp_code_mapping.csv 불러오기
(3965, 4)


,corp_code,corp_name_dart,stock_code,modify_date
0,00260985,한빛네트,036720,20170630
1,00264529,엔플렉스,040130,20170630
2,00358545,동서정보기술,055000,20170630
3,00231567,애드모바일,032600,20170630
4,00359614,리더컴,056140,20170630


## 5단계. company_master에 corp_code 붙이기
기업명 대신 `stock_code`를 기준으로 `master` 데이터와 DART 기업코드 데이터를 병합한다.  
중복 컬럼 문제를 방지하기 위해 기존 `corp_code` 관련 컬럼을 먼저 정리한다.

In [23]:
# =========================================================
# 5. company_master에 corp_code 붙이기
# =========================================================

for col in [
    "corp_code", "corp_code_x", "corp_code_y",
    "corp_name_dart", "corp_name_dart_x", "corp_name_dart_y"
]:
    if col in master.columns:
        master = master.drop(columns=[col])

master["stock_code"] = master["stock_code"].astype(str).str.zfill(6)
corp_df["stock_code"] = corp_df["stock_code"].astype(str).str.zfill(6)

master = master.merge(
    corp_df[["stock_code", "corp_code", "corp_name_dart"]],
    on="stock_code",
    how="left"
)

print(master["corp_code"].isna().sum(), "개 행에서 corp_code 누락")
display(master.head())

0 개 행에서 corp_code 누락


,company_name,stock_code,industry,fiscal_year,report_code,rcept_no,esg_source,esg_grade,e_grade,s_grade,g_grade,esg_year,esg_grade_num,e_grade_num,s_grade_num,g_grade_num,corp_code,corp_name_dart
0,삼성전자,005930,전기전자,2022,11011,NaN,한국ESG기준원,A,A,A+,B+,2023,4,4,5,3,00126380,삼성전자
1,삼성전자,005930,전기전자,2023,11011,NaN,한국ESG기준원,B+,B+,A,B,2024,3,3,4,2,00126380,삼성전자
2,삼성전자,005930,전기전자,2024,11011,NaN,한국ESG기준원,A,B+,A+,B+,2025,4,3,5,3,00126380,삼성전자
3,BYC,001460,섬유/의류,2022,11011,NaN,한국ESG기준원,D,D,D,D,2023,0,0,0,0,00122579,BYC
4,BYC,001460,섬유/의류,2023,11011,NaN,한국ESG기준원,D,D,D,C,2024,0,0,0,1,00122579,BYC


## 6단계. 사업보고서 rcept_no 찾기
`corp_code`와 `fiscal_year`를 이용해 해당 회계연도의 사업보고서 접수번호를 찾는다.  
수집한 `rcept_no`, `rcept_dt`, `report_nm`, `viewer_url`은 추후 재사용할 수 있도록 로그 파일로 저장한다.


In [24]:
# =========================================================
# 6. 사업보고서 rcept_no 찾기
# 체크포인트 저장 + 재사용 버전
# =========================================================

REPORT_LOG_PATH = os.path.join(SAVE_DIR, "dart_report_log.csv")

def find_annual_report(api_key, corp_code, fiscal_year):
    if pd.isna(corp_code):
        return {
            "rcept_no": None,
            "rcept_dt": None,
            "report_nm": None,
            "viewer_url": None,
            "collection_status": "missing_corp_code"
        }

    submit_year = int(fiscal_year) + 1

    params = {
        "crtfc_key": api_key,
        "corp_code": str(corp_code),
        "bgn_de": f"{submit_year}0101",
        "end_de": f"{submit_year}1231",
        "last_reprt_at": "Y",
        "pblntf_detail_ty": "A001",
        "page_count": "100"
    }

    try:
        res = requests.get(
            "https://opendart.fss.or.kr/api/list.json",
            params=params,
            timeout=20
        )

        payload = res.json()

    except Exception as e:
        return {
            "rcept_no": None,
            "rcept_dt": None,
            "report_nm": None,
            "viewer_url": None,
            "collection_status": f"api_error: {e}"
        }

    rows = payload.get("list", [])

    if not rows:
        return {
            "rcept_no": None,
            "rcept_dt": None,
            "report_nm": None,
            "viewer_url": None,
            "collection_status": "no_report_found"
        }

    target_year = str(int(fiscal_year))

    candidates = []

    for row in rows:
        report_nm = row.get("report_nm", "")
        if "사업보고서" in report_nm and target_year in report_nm:
            candidates.append(row)

    if len(candidates) == 0:
        for row in rows:
            report_nm = row.get("report_nm", "")
            if "사업보고서" in report_nm:
                candidates.append(row)

    if len(candidates) == 0:
        return {
            "rcept_no": None,
            "rcept_dt": None,
            "report_nm": None,
            "viewer_url": None,
            "collection_status": "annual_report_not_matched"
        }

    selected = candidates[0]
    rcept_no = selected.get("rcept_no")

    return {
        "rcept_no": rcept_no,
        "rcept_dt": selected.get("rcept_dt"),
        "report_nm": selected.get("report_nm"),
        "viewer_url": f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={rcept_no}",
        "collection_status": "success"
    }


if os.path.exists(REPORT_LOG_PATH):
    print("기존 dart_report_log.csv 불러오기")

    master_reports = pd.read_csv(
        REPORT_LOG_PATH,
        dtype={
            "stock_code": str,
            "corp_code": str,
            "rcept_no": str
        }
    )

    master_reports["stock_code"] = master_reports["stock_code"].astype(str).str.zfill(6)
    master_reports["corp_code"] = master_reports["corp_code"].astype(str).str.zfill(8)

else:
    print("DART 사업보고서 접수번호 새로 수집")

    drop_cols = [
        "rcept_no", "rcept_dt", "report_nm",
        "viewer_url", "collection_status"
    ]

    master_clean = master.drop(
        columns=[c for c in drop_cols if c in master.columns],
        errors="ignore"
    ).copy()

    report_rows = []

    for idx, row in master_clean.iterrows():
        result = find_annual_report(
            OPENDART_API_KEY,
            row.get("corp_code"),
            row.get("fiscal_year")
        )

        report_rows.append(result)

        if idx % 20 == 0:
            print(idx, "/", len(master_clean))

        time.sleep(0.15)

    report_df = pd.DataFrame(report_rows)

    master_reports = pd.concat(
        [master_clean.reset_index(drop=True), report_df.reset_index(drop=True)],
        axis=1
    )

    master_reports.to_csv(
        REPORT_LOG_PATH,
        index=False,
        encoding="utf-8-sig"
    )

print(master_reports.shape)
print(master_reports["collection_status"].value_counts(dropna=False))
display(master_reports.head())

기존 dart_report_log.csv 불러오기
(381, 22)
collection_status
success            373
no_report_found      8
Name: count, dtype: int64


,company_name,stock_code,industry,fiscal_year,report_code,esg_source,esg_grade,e_grade,s_grade,g_grade,esg_year,esg_grade_num,e_grade_num,s_grade_num,g_grade_num,corp_code,corp_name_dart,rcept_no,rcept_dt,report_nm,viewer_url,collection_status
0,삼성전자,005930,전기전자,2022,11011,한국ESG기준원,A,A,A+,B+,2023,4,4,5,3,00126380,삼성전자,20230307000542,20230307.0,사업보고서 (2022.12),https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20230307000542,success
1,삼성전자,005930,전기전자,2023,11011,한국ESG기준원,B+,B+,A,B,2024,3,3,4,2,00126380,삼성전자,20240312000736,20240312.0,사업보고서 (2023.12),https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20240312000736,success
2,삼성전자,005930,전기전자,2024,11011,한국ESG기준원,A,B+,A+,B+,2025,4,3,5,3,00126380,삼성전자,20250311001085,20250311.0,사업보고서 (2024.12),https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20250311001085,success
3,BYC,001460,섬유/의류,2022,11011,한국ESG기준원,D,D,D,D,2023,0,0,0,0,00122579,BYC,20230316001312,20230316.0,사업보고서 (2022.12),https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20230316001312,success
4,BYC,001460,섬유/의류,2023,11011,한국ESG기준원,D,D,D,C,2024,0,0,0,1,00122579,BYC,20240314001341,20240314.0,사업보고서 (2023.12),https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20240314001341,success


## 7단계. XML 다운로드 함수 정의
`rcept_no`를 이용해 OpenDART의 `document.xml` API에서 사업보고서 원문을 다운로드한다.  
ZIP 파일 안의 XML을 텍스트로 변환해 이후 ESG 문장 추출에 사용한다.

In [34]:
# =========================================================
# 7. XML 다운로드 함수
# 빠른 버전: XML 원문 Drive 저장 안 함
# =========================================================

import requests
import zipfile
from io import BytesIO
import pandas as pd

session = requests.Session()

def download_report_xml(api_key, rcept_no):
    if isinstance(rcept_no, pd.Series):
        rcept_no = rcept_no.iloc[0]

    if rcept_no is None:
        return None, "missing_rcept_no"

    rcept_no = str(rcept_no).strip()

    if rcept_no == "" or rcept_no.lower() == "nan":
        return None, "missing_rcept_no"

    params = {
        "crtfc_key": api_key,
        "rcept_no": rcept_no
    }

    try:
        res = session.get(
            "https://opendart.fss.or.kr/api/document.xml",
            params=params,
            timeout=20
        )

        res.raise_for_status()

        with zipfile.ZipFile(BytesIO(res.content)) as zf:
            first_name = zf.namelist()[0]
            xml_text = zf.read(first_name).decode("utf-8", errors="ignore")

        return xml_text, "success"

    except Exception as e:
        return None, f"xml_error: {e}"

## 8단계. ESG seed dictionary 및 passage 추출 함수 정의
환경(E), 사회(S), 지배구조(G)별 seed 단어를 정의한다.  
문장 안에 포함된 seed 단어를 기준으로 ESG 관련 문장을 판별하고, 해당 문장의 E/S/G 차원을 분류한다.

In [35]:
# =========================================================
# 8. ESG seed dictionary 및 passage 추출 함수
# =========================================================

E_SEED = [
    "탄소", "온실가스", "탄소중립", "넷제로", "재생에너지",
    "에너지", "전력", "폐기물", "재활용", "폐수"
]

S_SEED = [
    "안전", "산업재해", "중대재해", "임직원", "노동",
    "인권", "교육훈련", "협력사", "공급망", "지역사회"
]

G_SEED = [
    "이사회", "사외이사", "감사위원회", "독립성", "윤리",
    "준법", "컴플라이언스", "부패방지", "주주", "의결권"
]

seed_dict = {
    "E": E_SEED,
    "S": S_SEED,
    "G": G_SEED
}

all_seed_terms = E_SEED + S_SEED + G_SEED


def clean_xml_to_text(xml_text):
    soup = BeautifulSoup(xml_text, "lxml")
    text = soup.get_text(separator=" ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_sentences(text):
    text = str(text)

    sents = re.split(r"(?<=[다요음임함됨\.])\s+", text)

    sents = [
        s.strip()
        for s in sents
        if len(s.strip()) >= 20
    ]

    return sents


def detect_dimension(sentence):
    matched = {}

    for dim, terms in seed_dict.items():
        hits = [term for term in terms if term in sentence]

        if len(hits) > 0:
            matched[dim] = hits

    if len(matched) == 0:
        return None, []

    dim = max(matched.keys(), key=lambda k: len(matched[k]))
    terms = matched[dim]

    return dim, terms


def detect_section(sentence):
    section_keywords = {
        "II": ["II. 사업의 내용", "Ⅱ. 사업의 내용", "사업의 내용"],
        "IV": ["IV. 이사의 경영진단", "Ⅳ. 이사의 경영진단", "경영진단"],
        "VI": ["VI. 이사회", "Ⅵ. 이사회", "이사회 등 회사의 기관"]
    }

    for section, keywords in section_keywords.items():
        for keyword in keywords:
            if keyword in sentence:
                return section

    return "II_IV_VI_candidate"


def extract_esg_passages(xml_text):
    text = clean_xml_to_text(xml_text)
    sentences = split_sentences(text)

    rows = []

    for sent in sentences:
        dim, terms = detect_dimension(sent)

        if dim is not None:
            rows.append({
                "section": detect_section(sent),
                "passage": sent,
                "dimension": dim,
                "matched_terms": ",".join(terms)
            })

    return pd.DataFrame(rows)


sample_success = master_reports[
    master_reports["collection_status"] == "success"
].head(1)

if len(sample_success) > 0:
    sample_rcept = sample_success.iloc[0]["rcept_no"]

    xml_text, status = download_report_xml(
        OPENDART_API_KEY,
        sample_rcept
    )

    print("테스트 rcept_no:", sample_rcept)
    print("XML 상태:", status)

    if xml_text is not None:
        sample_passages = extract_esg_passages(xml_text)
        print("추출 passage 수:", len(sample_passages))
        display(sample_passages.head())

테스트 rcept_no: 20230307000542
XML 상태: success


/tmp/ipykernel_475/3005066530.py:30: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(xml_text, "lxml")


추출 passage 수: 397


,section,passage,dimension,matched_terms
0,VI,이사회 등 회사의 기관에 관한 사항 --------------------------------- 356 1.,G,이사회
1,II_IV_VI_candidate,이사회에 관한 사항 --------------------------------- 356 2.,G,이사회
2,II_IV_VI_candidate,주주총회 등에 관한 사항 --------------------------------- 380 VII.,G,주주
3,II_IV_VI_candidate,주주에 관한 사항 --------------------------------- 385 1.,G,주주
4,II_IV_VI_candidate,최대주주 및 그 특수관계인의 주식소유 현황 --------------------------------- 385 2.,G,주주


## 9단계. 전체 기업-연도 ESG passage 추출
각 기업-연도의 사업보고서 XML에서 ESG 관련 문장을 추출한다.  
ESG seed 단어가 2개 이상 포함된 문장만 남겨 잡음을 줄였고, 병렬 처리와 체크포인트 저장을 통해 실행 효율성을 높였다.

In [36]:
# =========================================================
# 9. 전체 기업-연도 passage 추출
# 병렬 처리 + Fast Strict + checkpoint 저장 버전
# =========================================================

import os
import re
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import clear_output

PASSAGE_PATH = os.path.join(SAVE_DIR, "passage_df_checkpoint_v2_parallel.csv")
STATUS_PATH = os.path.join(SAVE_DIR, "xml_status_log_v2_parallel.csv")

MAX_WORKERS = 3   # 너무 높이면 DART 서버/API 오류 가능


def extract_esg_passages_fast(xml_text):
    if xml_text is None:
        return pd.DataFrame()

    text = str(xml_text)

    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("&nbsp;", " ").replace("&amp;", " ")
    text = re.sub(r"\s+", " ", text).strip()

    sentences = re.split(r"(?<=[다요음임함됨\.])\s+", text)

    rows = []

    for sent in sentences:
        sent = sent.strip()

        if len(sent) < 40:
            continue

        if len(sent) > 500:
            continue

        matched = {}
        total_hit_count = 0

        for dim, terms in seed_dict.items():
            hits = [term for term in terms if term in sent]

            if len(hits) > 0:
                matched[dim] = hits
                total_hit_count += len(hits)

        if total_hit_count < 2:
            continue

        dim = max(matched.keys(), key=lambda k: len(matched[k]))
        terms = matched[dim]

        rows.append({
            "section": "ESG_candidate_fast_parallel",
            "passage": sent,
            "dimension": dim,
            "matched_terms": ",".join(terms)
        })

    return pd.DataFrame(rows)


# 기존 checkpoint 불러오기
if os.path.exists(PASSAGE_PATH):
    passage_df = pd.read_csv(
        PASSAGE_PATH,
        dtype={"stock_code": str, "corp_code": str, "rcept_no": str}
    )
    completed_rcepts = set(passage_df["rcept_no"].astype(str).unique())
else:
    passage_df = pd.DataFrame()
    completed_rcepts = set()


if os.path.exists(STATUS_PATH):
    status_df = pd.read_csv(STATUS_PATH, dtype={"rcept_no": str})
    completed_status_rcepts = set(status_df["rcept_no"].astype(str).unique())
else:
    status_df = pd.DataFrame()
    completed_status_rcepts = set()


# 처리 대상 만들기
target_rows = []

for _, row in master_reports.iterrows():
    rcept_no = str(row.get("rcept_no")).strip()

    if rcept_no == "" or rcept_no.lower() == "nan":
        continue

    if rcept_no in completed_rcepts or rcept_no in completed_status_rcepts:
        continue

    target_rows.append(row.to_dict())

print("이번 실행 처리 대상 수:", len(target_rows))


def process_one(row):
    rcept_no = str(row.get("rcept_no")).strip()

    xml_text, xml_status = download_report_xml(
        OPENDART_API_KEY,
        rcept_no
    )

    status_row = {
        "company_name": row.get("company_name"),
        "stock_code": row.get("stock_code"),
        "corp_code": row.get("corp_code"),
        "fiscal_year": row.get("fiscal_year"),
        "esg_year": row.get("esg_year"),
        "rcept_no": rcept_no,
        "xml_status": xml_status
    }

    if xml_text is None:
        return None, status_row

    passages = extract_esg_passages_fast(xml_text)

    if len(passages) == 0:
        return None, status_row

    passages["company_name"] = row.get("company_name")
    passages["stock_code"] = row.get("stock_code")
    passages["corp_code"] = row.get("corp_code")
    passages["fiscal_year"] = row.get("fiscal_year")
    passages["esg_year"] = row.get("esg_year")
    passages["rcept_no"] = rcept_no
    passages["rcept_dt"] = row.get("rcept_dt")
    passages["viewer_url"] = row.get("viewer_url")

    return passages, status_row


start_time = time.time()

new_passages = []
new_status = []

processed_count = 0
success_xml_count = 0
fail_xml_count = 0
passage_success_count = 0
new_passage_count = 0


with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(process_one, row) for row in target_rows]

    for future in as_completed(futures):
        passages, status_row = future.result()

        processed_count += 1
        new_status.append(status_row)

        if str(status_row["xml_status"]).startswith("success"):
            success_xml_count += 1
        else:
            fail_xml_count += 1

        if passages is not None:
            passage_success_count += 1
            new_passage_count += len(passages)
            new_passages.append(passages)

        # 10개마다 저장
        if processed_count % 10 == 0:
            if len(new_passages) > 0:
                temp_passage = pd.concat(new_passages, ignore_index=True)

                passage_df = pd.concat(
                    [passage_df, temp_passage],
                    ignore_index=True
                )

                passage_df.to_csv(
                    PASSAGE_PATH,
                    index=False,
                    encoding="utf-8-sig"
                )

                new_passages = []

            if len(new_status) > 0:
                temp_status = pd.DataFrame(new_status)

                status_df = pd.concat(
                    [status_df, temp_status],
                    ignore_index=True
                )

                status_df.to_csv(
                    STATUS_PATH,
                    index=False,
                    encoding="utf-8-sig"
                )

                new_status = []

            clear_output(wait=True)

            elapsed = round((time.time() - start_time) / 60, 1)

            print("9단계 진행 중 - Parallel 버전")
            print(f"- 전체 행 수: {len(master_reports)}")
            print(f"- 이번 실행 처리 대상 수: {len(target_rows)}")
            print(f"- 처리 완료 수: {processed_count}")
            print(f"- XML 성공: {success_xml_count}")
            print(f"- XML 실패: {fail_xml_count}")
            print(f"- passage 추출 성공 firm-year 수: {passage_success_count}")
            print(f"- 새로 추출된 passage 수: {new_passage_count}")
            print(f"- 현재 passage_df shape: {passage_df.shape}")
            print(f"- 경과 시간: {elapsed}분")


# 남은 데이터 저장
if len(new_passages) > 0:
    temp_passage = pd.concat(new_passages, ignore_index=True)

    passage_df = pd.concat(
        [passage_df, temp_passage],
        ignore_index=True
    )

if len(new_status) > 0:
    temp_status = pd.DataFrame(new_status)

    status_df = pd.concat(
        [status_df, temp_status],
        ignore_index=True
    )


# 중복 제거
if len(passage_df) > 0:
    passage_df = passage_df.drop_duplicates(
        subset=["rcept_no", "passage"]
    )

if len(status_df) > 0:
    status_df = status_df.drop_duplicates(
        subset=["rcept_no"],
        keep="last"
    )


passage_df.to_csv(
    PASSAGE_PATH,
    index=False,
    encoding="utf-8-sig"
)

status_df.to_csv(
    STATUS_PATH,
    index=False,
    encoding="utf-8-sig"
)


clear_output(wait=True)

elapsed = round((time.time() - start_time) / 60, 1)

print("9단계 완료 - Parallel 버전")
print("=" * 50)
print(f"전체 master_reports 행 수: {len(master_reports)}")
print(f"이번 실행 처리 대상 수: {len(target_rows)}")
print(f"처리 완료 수: {processed_count}")
print(f"XML 다운로드 성공 수: {success_xml_count}")
print(f"XML 다운로드 실패 수: {fail_xml_count}")
print(f"passage 추출 성공 firm-year 수: {passage_success_count}")
print(f"이번 실행에서 새로 추출된 passage 수: {new_passage_count}")
print(f"최종 passage_df shape: {passage_df.shape}")
print(f"최종 status_df shape: {status_df.shape}")
print(f"소요 시간: {elapsed}분")
print("=" * 50)

display(passage_df.head())

9단계 완료 - Parallel 버전
전체 master_reports 행 수: 381
이번 실행 처리 대상 수: 373
처리 완료 수: 373
XML 다운로드 성공 수: 365
XML 다운로드 실패 수: 8
passage 추출 성공 firm-year 수: 365
이번 실행에서 새로 추출된 passage 수: 14065
최종 passage_df shape: (13610, 12)
최종 status_df shape: (373, 7)
소요 시간: 56.4분


,section,passage,dimension,matched_terms,company_name,stock_code,corp_code,fiscal_year,esg_year,rcept_no,rcept_dt,viewer_url
0,ESG_candidate_fast_parallel,경영진의 중요한 변동 당사의 이사회는 주주총회에서 선임한 이사로 구성되며 회사 업무의 중요사항을 결의합니다.,G,"이사회,주주",삼성전자,005930,00126380,2023,2024,20240312000736,20240312.0,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20240312000736
1,ESG_candidate_fast_parallel,"2023년말 현재 당사의 이사회는 사내이사 5인(한종희, 경계현, 노태문, 박학규,이정배)과 사외이사 6인(김한조, 김선욱, 김종훈, 김준성, 허은녕, 유명희) 등 총 11인의 이사로 구성되어 있습니다.",G,"이사회,사외이사",삼성전자,005930,00126380,2023,2024,20240312000736,20240312.0,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20240312000736
2,ESG_candidate_fast_parallel,또한 2022년 3월 16일 주주총회에서 신규 선임된 경계현 사내이사가 당일 이사회를 통해 대표이사로 선임되었습니다.,G,"이사회,주주",삼성전자,005930,00126380,2023,2024,20240312000736,20240312.0,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20240312000736
3,ESG_candidate_fast_parallel,"※ 2022년 상반기에 한화진 사외이사가 사임하고, 박병국 사외이사가 퇴임하여 2022년 11월 3일 임시주주총회에서 허은녕 사외이사, 유명희 사외이사가 신규 선임되었습니다.※ 2023년 3월 15일 주주총회...",G,"이사회,사외이사,주주",삼성전자,005930,00126380,2023,2024,20240312000736,20240312.0,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20240312000736
4,ESG_candidate_fast_parallel,"(기준일 : 2023년 12월 31일 ) (단위 : 주, %) 변동일 최대주주명 소유주식수 지분율 변동원인 비 고 2021.04.29 삼성생명보험㈜ 1,263,050,053 21.16% 변동전 최대주주의 피상...",G,"주주,의결권",삼성전자,005930,00126380,2023,2024,20240312000736,20240312.0,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20240312000736


### DART XML 수집 및 ESG Passage 추출 결과

- 전체 381개 기업-연도 중 373개 행을 대상으로 DART XML 원문 수집을 수행함

- XML 다운로드 및 ESG passage 추출에 성공한 기업-연도는 총 365개였으며, 8개 기업-연도는 XML 다운로드에 실패함

- XML 실패 원인은 OpenDART 서버 응답 지연, 일시적 API 오류, 원문 ZIP 파일 구조 문제, 정정공시 및 보고서 형식 차이 등으로 판단함

- XML 수집 실패는 실제 ESG 정보가 존재하지 않는다는 의미와 다르므로, 실패 행의 텍스트 점수를 임의로 0으로 대체하지 않음

- XML 다운로드 상태 및 실패 사유는 `xml_status_log` 파일에 별도로 기록하여 데이터 lineage를 유지함

- 최종적으로 365개 기업-연도에서 총 13,610개의 ESG 관련 passage를 확보함

- 이후 동일 기업-연도에서 추출된 여러 ESG passage를 하나의 document로 통합하여 TF-IDF 및 형태소 분석 기반 feature 생성에 활용함

## 10단계. firm-year 단위 document 만들기
같은 기업-연도에서 추출된 여러 ESG passage를 하나의 문서로 통합한다.  
이 단계에서 기업-연도당 1행 구조의 `doc_df`를 만들어 이후 TF-IDF 분석의 입력 데이터로 사용한다.

In [37]:
# =========================================================
# 10. firm-year 단위 document 만들기
# =========================================================

group_cols = [
    "company_name", "stock_code", "corp_code",
    "fiscal_year", "esg_year", "rcept_no", "rcept_dt", "viewer_url"
]

doc_df = (
    passage_df
    .groupby(group_cols, dropna=False)
    .agg(
        document=("passage", lambda x: " ".join(x)),
        passage_count=("passage", "count"),
        matched_terms_all=("matched_terms", lambda x: ",".join(x))
    )
    .reset_index()
)

doc_df["total_char_count"] = doc_df["document"].str.len()

grade_merge_cols = ["stock_code", "fiscal_year", "esg_year"]

grade_use_cols = grade_merge_cols + [
    c for c in master.columns
    if "grade" in c
]

doc_df = doc_df.merge(
    master[grade_use_cols].drop_duplicates(),
    on=grade_merge_cols,
    how="left"
)

print(doc_df.shape)
display(doc_df.head())

(365, 20)


,company_name,stock_code,corp_code,fiscal_year,esg_year,rcept_no,rcept_dt,viewer_url,document,passage_count,matched_terms_all,total_char_count,esg_grade,e_grade,s_grade,g_grade,esg_grade_num,e_grade_num,s_grade_num,g_grade_num
0,BGF리테일,282330,01263022,2022,2023,20230320000886,20230320.0,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20230320000886,2019년 03월 27일 정기주총 사내이사 류왕선사외이사 백복현사외이사 한명관사외이사 임영철 - 감사 전홍(사임) 2020년 03월 25일 정기주총 대표이사 이건준기타비상무이사 홍정국 사외이사 김난도 대표이사...,41,"사외이사,주주,감사위원회,주주,의결권,이사회,주주,이사회,주주,이사회,주주,이사회,주주,이사회,사외이사,이사회,사외이사,감사위원회,사외이사,감사위원회,주주,이사회,사외이사,독립성,주주,이사회,독립성,주주,이사...",6903,A,A,A+,A,4,4,5,4
1,BGF리테일,282330,01263022,2023,2024,20240412003349,20240412.0,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20240412003349,2019년 03월 27일 정기주총 사내이사 류왕선사외이사 백복현사외이사 한명관사외이사 임영철 - 감사 전홍(사임) 2020년 03월 25일 정기주총 대표이사 이건준기타비상무이사 홍정국 사외이사 김난도 대표이사...,54,"이사회,사외이사,주주,이사회,주주,이사회,주주,이사회,주주,이사회,주주,안전,안전,안전,감사위원회,주주,의결권,이사회,주주,이사회,주주,이사회,주주,이사회,주주,이사회,주주,이사회,사외이사,이사회,사외이사,감...",8442,A,A,A+,A,4,4,5,4
2,BGF리테일,282330,01263022,2024,2025,20250318000733,20250318.0,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20250318000733,"또한, 우리는 독립성 관련 윤리적 요구사항들을 준수하고, 우리의 독립성 문제와 관련된다고 판단되는 모든 관계와 기타사항들 및 해당되는 경우 관련 제도적 안전장치를 지배기구와 커뮤니케이션한다는 진술을 지배기구에...",4,"독립성,윤리,이사회,주주,이사회,주주,감사위원회,독립성",999,A+,A+,A+,A,5,5,5,4
3,BNK금융지주,138930,00858364,2022,2023,20230309000759,20230309.0,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20230309000759,"관련 감사위원회의 역할 변경 대외법률 시행 및 개정내용 반영,임원 임기 운영 관련 내부 원칙 반영 등 2021.03.26 제10기 정기주주총회 ①신주의 배당기산일 간주 규정 및 준용 규정 삭제②전자등록제도 도...",54,"이사회,감사위원회,주주,임직원,지역사회,탄소,이사회,주주,재생에너지,에너지,이사회,주주,임직원,이사회,사외이사,이사회,독립성,이사회,사외이사,사외이사,주주,이사회,사외이사,이사회,감사위원회,이사회,감사위원회,...",12350,A,A,A+,A,4,4,5,4
4,BNK금융지주,138930,00858364,2023,2024,20240314001564,20240314.0,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=20240314001564,"214 독립된 감사인의 감사보고서 주식회사 BNK금융지주와 그 종속기업 주주 및 이사회 귀중 감사의견 우리는 주식회사 BNK금융지주 및 종속기업들(이하 ""연결회사"")의 연결재무제표를 감사하였습니다. 또한 우리...",5,"이사회,주주,독립성,윤리,이사회,주주,이사회,주주,감사위원회,독립성",1044,A,A+,A+,A,4,5,5,4


## 11단계. Kiwi 형태소 분석기 준비 및 불용어 사전 구성
Kiwi 형태소 분석기를 초기화하고 불용어 사전을 구성한다.  
일반 불용어, 공시 반복어, 회사명, DART 회사명을 제거 대상으로 포함해 분석에 불필요한 단어를 줄인다.

In [38]:
# =========================================================
# 11. Kiwi 형태소 분석기 준비 + 불용어 사전 구성
# =========================================================

from kiwipiepy import Kiwi
from collections import Counter
import re
import pandas as pd

kiwi = Kiwi()

# 기본 불용어
BASE_STOPWORDS = set("""
및 관련 통해 대한 당사 회사 사업 보고서 내용 경우 등 수 년 월 일 현재 기준 해당
있습니다 있습니다 대한 것으로 또한 위하여 따라 이를 또는 이러한 그리고 그러나
관리 실시 제공 추진 확대 강화 구축 운영 포함 주요 기타 제 관한 각
전자 주식회사 연결 재무제표 감사 법률 규정 사항
""".split())

# 공시에서 너무 자주 나오는 일반 단어 후보
DISCLOSURE_STOPWORDS = set("""
제출 공시 정정 첨부 서류 법인 기업 그룹 계열 대상 현황
기준일 당기 전기 전년 말 연결 별도 재무 영업 자산 부채 손익
주석 표 참조 항목 단위 백만원 천원 원
""".split())

# 회사명 불용어 추가
if "company_name" in master.columns:
    company_stopwords = set(master["company_name"].dropna().astype(str).tolist())
else:
    company_stopwords = set()

# DART 회사명도 추가
if "corp_name_dart" in master_reports.columns:
    dart_company_stopwords = set(master_reports["corp_name_dart"].dropna().astype(str).tolist())
else:
    dart_company_stopwords = set()

STOPWORDS = BASE_STOPWORDS | DISCLOSURE_STOPWORDS | company_stopwords | dart_company_stopwords

print("불용어 개수:", len(STOPWORDS))

불용어 개수: 216


## 12단계. Kiwi 형태소 분석 기반 전처리
문서를 정규화한 뒤 Kiwi 형태소 분석을 수행한다.  
명사류와 외국어 토큰만 남기고, 불용어·짧은 단어·숫자·시간 표현·코드성 표현을 제거하여 TF-IDF 직전의 `tokenized_text`를 생성한다.

### [kiwi 사용 근거]

- Kiwi 형태소 분석기를 사용하여 한국어 문서를 형태소 단위로 분리
- 사업보고서 특성상 조사, 어미, 공시 반복 표현이 많기 때문에 단순 공백 기반 토큰화보다 형태소 분석 기반 전처리가 적합하다고 판단함

- ESG 분석에 핵심적인 명사 중심 토큰만 추출하여 TF-IDF 및 FastText 분석의 품질을 높이고자 함

- Kiwi는 비교적 빠른 속도와 안정적인 명사 추출 성능을 제공하며, 대규모 DART 사업보고서 corpus 처리에 적합하다고 판단하여 사용함

- 또한 회사명, 공시 반복 표현, 숫자, 코드성 표현 등을 제거하여 noise를 줄이고 ESG 관련 정보 밀도를 높이고자 함

In [39]:
# =========================================================
# 12. Kiwi 형태소 분석 기반 전처리
# TF-IDF 직전 최종 전처리 단계
# =========================================================

# ---------------------------------------------------------
# 12-1. 문서 1차 정규화
# ---------------------------------------------------------

def normalize_text(text):
    """
    공시 원문에서 형태소 분석 전에 불필요한 기호와 반복 공백을 정리.
    단, ESG 핵심어가 손실되지 않도록 한글/영문/숫자는 유지.
    """

    text = str(text)

    # HTML 특수문자 일부 정리
    text = text.replace("&nbsp;", " ")
    text = text.replace("&amp;", " ")

    # 괄호 안 긴 법률/표 번호 등 일부 완화
    text = re.sub(r"\[[^\]]{1,30}\]", " ", text)
    text = re.sub(r"\([0-9가-힣A-Za-z\s,\.-]{1,30}\)", " ", text)

    # 한글, 영문, 숫자, 공백 외 제거
    text = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", text)

    # 반복 공백 제거
    text = re.sub(r"\s+", " ", text).strip()

    return text


# ---------------------------------------------------------
# 12-2. 토큰 필터링 규칙
# ---------------------------------------------------------

ALLOWED_POS_PREFIX = (
    "N",   # NNG 일반명사, NNP 고유명사 등
)

ALLOWED_POS_EXACT = {
    "SL",  # 외국어: ESG, RE100 등
}

# 너무 일반적인 단일 토큰 제거
EXTRA_BAD_TOKENS = set("""
가능 필요 예정 목표 계획 방안 활동 업무 부문 부서 분야 방식 정도
사항 내용 관련 대상 기준 결과 통해 위해 따른 따라
""".split())

STOPWORDS = STOPWORDS | EXTRA_BAD_TOKENS


def is_valid_token(form, tag):
    """
    Kiwi 분석 결과에서 사용할 토큰인지 판단.
    """

    form = str(form).strip()
    tag = str(tag).strip()

    if form == "":
        return False

    # 품사 제한: 명사류 + 외국어만 사용
    if not (tag.startswith(ALLOWED_POS_PREFIX) or tag in ALLOWED_POS_EXACT):
        return False

    # 불용어 제거
    if form in STOPWORDS:
        return False

    # 너무 짧은 토큰 제거
    if len(form) < 2:
        return False

    # 숫자만 있는 토큰 제거
    if form.isdigit():
        return False

    # 2024년, 12월 같은 시간 표현 제거
    if re.fullmatch(r"\d{2,4}년?", form):
        return False

    if re.fullmatch(r"\d{1,2}월", form):
        return False

    # 영문 한 글자 제거
    if re.fullmatch(r"[A-Za-z]", form):
        return False

    # 반복 숫자/코드성 표현 제거
    if re.fullmatch(r"[0-9A-Za-z]{6,}", form):
        return False

    return True


# ---------------------------------------------------------
# 12-3. Kiwi 토큰화 함수
# ---------------------------------------------------------

def tokenize_with_kiwi(text):
    """
    Kiwi 형태소 분석기를 사용해 명사 중심 토큰을 추출.
    """

    text = normalize_text(text)

    analyzed = kiwi.tokenize(text)

    tokens = []

    for token in analyzed:
        form = token.form
        tag = token.tag

        if is_valid_token(form, tag):
            tokens.append(form)

    return tokens


# ---------------------------------------------------------
# 12-4. doc_df 전체에 적용
# ---------------------------------------------------------

doc_df["tokens"] = doc_df["document"].apply(tokenize_with_kiwi)
doc_df["tokenized_text"] = doc_df["tokens"].apply(lambda x: " ".join(x))
doc_df["total_word_count"] = doc_df["tokens"].apply(len)
doc_df["unique_word_count"] = doc_df["tokens"].apply(lambda x: len(set(x)))


display(doc_df[[
    "company_name",
    "fiscal_year",
    "passage_count",
    "total_word_count",
    "unique_word_count",
    "tokenized_text"
]].head())

,company_name,fiscal_year,passage_count,total_word_count,unique_word_count,tokenized_text
0,BGF리테일,2022,41,1097,281,정기 주총 사내 이사 류왕선 사외이사 백복현 사외 이사 명관 사외 이사 전홍 정기 주총 대표 이사 이건준 상무이사 홍정국 사외이사 김난도 대표 이사 재구 사내 이사 홍정국 정기 주총 사외이사 최자원 사외이사 ...
1,BGF리테일,2023,54,1314,323,정기 주총 사내 이사 류왕선 사외이사 백복현 사외 이사 명관 사외 이사 전홍 정기 주총 대표 이사 이건준 상무이사 홍정국 사외이사 김난도 대표 이사 재구 사내 이사 홍정국 정기 주총 사외이사 최자원 사외이사 ...
2,BGF리테일,2024,4,134,59,우리 독립 윤리 요구 준수 우리 독립 문제 판단 관계 제도 안전 장치 지배 기구 커뮤니케이션 진술 지배 기구 작성 외부 국제 회계 위원회 국제 회계 채택 회계 처리 한국 채택 국제 회계 작성 회계 지배 관계 ...
3,BNK금융지주,2022,54,2091,568,위원회 역할 변경 대외 시행 개정 반영 임원 임기 내부 원칙 반영 정기 주주 총회 신주 배당 기사 간주 준용 삭제 등록 제도 도입 반영 주주 명부 폐쇄 기간 삭제 자구 수정 이사회 ESG 위원회 신설 상법 상...
4,BNK금융지주,2023,5,166,65,독립 감사인 BNK 금융 지주 종속 주주 이사회 의견 우리 BNK 금융 지주 종속 이하 우리 독립 윤리 요구 준수 우리 독립 문제 판단 관계 제도 안전 장치 지배 기구 커뮤니케이션 진술 지배 기구 비교 표시 ...


## 12-5단계. 전처리 품질 점검
전체 토큰 빈도와 문서별 토큰 수를 확인한다.  
상위 토큰에 의미 없는 단어가 많이 남아 있는지, 토큰 수가 지나치게 적은 문서가 있는지 점검한다.

In [40]:
# =========================================================
# 12-5. 전처리 품질 점검
# =========================================================

# 전체 토큰 빈도 확인
all_tokens = []

for tokens in doc_df["tokens"]:
    all_tokens.extend(tokens)

token_freq = Counter(all_tokens)

token_freq_df = pd.DataFrame(
    token_freq.most_common(100),
    columns=["token", "count"]
)

display(token_freq_df.head(50))


# 문서별 토큰 수 분포 확인
display(doc_df[["total_word_count", "unique_word_count"]].describe())


# 토큰 수가 너무 적은 문서 확인
low_token_docs = doc_df[doc_df["total_word_count"] < 10].copy()

print("토큰 수 10개 미만 문서 수:", len(low_token_docs))

display(low_token_docs[[
    "company_name",
    "fiscal_year",
    "passage_count",
    "total_word_count",
    "document",
    "tokenized_text"
]].head(10))

,token,count
0,이사,15836
1,주주,10104
2,위원회,9195
3,사외,7916
4,이사회,7722
5,총회,5491
6,찬성,4360
7,선임,4284
8,위원,3699
9,주식,3654


,total_word_count,unique_word_count
count,365.000000,365.000000
mean,981.512329,306.734247
std,911.212487,228.363331
min,26.000000,18.000000
25%,182.000000,99.000000
50%,819.000000,287.000000
75%,1391.000000,433.000000
max,6099.000000,1302.000000


토큰 수 10개 미만 문서 수: 0


,company_name,fiscal_year,passage_count,total_word_count,document,tokenized_text


## 12-6단계. ESG seed 단어 보존 여부 확인
전처리 후에도 ESG seed 단어가 실제 토큰으로 남아 있는지 확인한다.  
주요 seed 단어가 모두 사라졌다면 전처리 규칙이 너무 강한 것이므로 수정이 필요하다.

In [41]:
# =========================================================
# 12-6. ESG seed 단어가 전처리 후 살아있는지 확인
# =========================================================

seed_check_rows = []

for dim, terms in seed_dict.items():
    for term in terms:
        count = token_freq.get(term, 0)

        seed_check_rows.append({
            "dimension": dim,
            "seed_term": term,
            "token_count_after_preprocessing": count
        })

seed_check_df = pd.DataFrame(seed_check_rows)

display(seed_check_df)


# seed 단어 중 전처리 후 한 번도 안 나온 단어
missing_seed_df = seed_check_df[
    seed_check_df["token_count_after_preprocessing"] == 0
]

print("전처리 후 등장하지 않은 seed 단어 수:", len(missing_seed_df))
display(missing_seed_df)

,dimension,seed_term,token_count_after_preprocessing
0,E,탄소,993
1,E,온실가스,1052
2,E,탄소중립,0
3,E,넷제로,0
4,E,재생에너지,0
5,E,에너지,2154
6,E,전력,376
7,E,폐기물,492
8,E,재활용,0
9,E,폐수,118


전처리 후 등장하지 않은 seed 단어 수: 11


,dimension,seed_term,token_count_after_preprocessing
2,E,탄소중립,0
3,E,넷제로,0
4,E,재생에너지,0
8,E,재활용,0
11,S,산업재해,0
12,S,중대재해,0
16,S,교육훈련,0
19,S,지역사회,0
23,G,독립성,0
26,G,컴플라이언스,0


## 12-7단계. 전처리 결과 저장
최종 전처리된 `doc_df`를 CSV 파일로 저장한다.  
이 파일은 이후 TF-IDF, FastText, 회귀분석 단계에서 바로 불러와 사용할 수 있다.

In [42]:
# =========================================================
# 12-7. 전처리 결과 저장
# =========================================================

PREPROCESSED_PATH = os.path.join(SAVE_DIR, "doc_df_preprocessed_kiwi_v2.csv")

doc_df.to_csv(
    PREPROCESSED_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("전처리 결과 저장 완료:")
print(PREPROCESSED_PATH)

전처리 결과 저장 완료:
/content/drive/MyDrive/esg_dart_project_v2/doc_df_preprocessed_kiwi_v2.csv


## TF-IDF 단계 전 확인 사항
- `passage_df`에 ESG 관련 문장이 적절히 추출되었는지 확인한다.
- `doc_df`가 기업-연도당 1행 구조인지 확인한다.
- `token_freq_df` 상위 토큰에 회사명이나 공시 형식어가 과도하게 남아 있지 않은지 확인한다.
- `seed_check_df`에서 주요 ESG seed 단어가 전처리 후에도 살아 있는지 확인한다.